# 05 · Retrieve and rerank

Stages 5 and 6 of the pipeline (see `ARCHITECTURE.md`): the first half of answering a question.

1. **Retrieve**: turn the question into a vector and take the **20** closest chunks from Chroma. Fast, but it only compares vectors made separately.
2. **Rerank**: a **cross-encoder** reads the question and each of those 20 chunks *together* and scores how well the chunk answers it. Keep the best **5**.

Both strategies (`acre_fixed`, `acre_semantic`) are searched, so they can be compared.

⚠️ Memory: peaks at ~850 MB RAM (all-MiniLM ~700 MB as in `04`, plus the reranker ~150 MB). The first run downloads the reranker once (~90 MB). Restart other notebook kernels first if RAM is tight.

Reads `data/chroma_db/` (built by `04_embed_store`). Writes nothing.

In [1]:
import os
from pathlib import Path

# Hugging Face warns that Windows can't use symlinks in its model cache; caching still works.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import pandas as pd
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

CHROMA_DIR = Path("../data/chroma_db")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
STRATEGIES = ["fixed", "semantic"]
RETRIEVE_K = 20
KEEP = 5

pd.set_option("display.max_colwidth", 110)

# Must be the same model that built the index, or question and chunk vectors won't be comparable.
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, encode_kwargs={"normalize_embeddings": True})
stores = {
    name: Chroma(collection_name=f"acre_{name}", embedding_function=embeddings, persist_directory=str(CHROMA_DIR))
    for name in STRATEGIES
}
for name, store in stores.items():
    print(f"acre_{name}: {store._collection.count()} vectors")

reranker = CrossEncoder(RERANK_MODEL)
print("Reranker loaded:", RERANK_MODEL)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

acre_fixed: 324 vectors
acre_semantic: 330 vectors


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


## The two steps

`search()` runs both steps for one question and one strategy, and returns all 20 retrieved chunks with:

- `retrieve_rank` and `distance`: order by vector distance (lower distance = closer), as in `04`
- `rerank_rank` and `rerank_score`: order by the cross-encoder (**higher** score = more relevant)

The cross-encoder reads up to 512 tokens of question + chunk, so unlike the embedding model (256), it sees all of nearly every chunk.

In [2]:
def search(question, strategy):
    hits = stores[strategy].similarity_search_with_score(question, k=RETRIEVE_K)
    scores = reranker.predict([(question, doc.page_content) for doc, _ in hits])

    results = pd.DataFrame([
        {
            "retrieve_rank": i,
            "distance": round(distance, 3),
            "rerank_score": round(float(score), 3),
            "chunk_id": doc.metadata["chunk_id"],
            "type": doc.metadata["type"],
            "section": doc.metadata["section"],
            "text": doc.page_content,
        }
        for i, ((doc, distance), score) in enumerate(zip(hits, scores), start=1)
    ])
    results = results.sort_values("rerank_score", ascending=False).reset_index(drop=True)
    results.insert(0, "rerank_rank", results.index + 1)
    results["preview"] = results["text"].str[:110].str.replace("\n", " ")
    return results


def show_chunk(results, rank):
    """Print the full text of the chunk at this rerank rank."""
    row = results.iloc[rank - 1]
    print(f"=== rerank #{rank} · retrieve #{row.retrieve_rank} · {row.chunk_id} · {row.section} ===")
    print(row.text)

## Before vs. after: one question

The question from `04`. There, plain retrieval put a *definition* table first, and the chunk with the actual answer ("approximately $8.6 million" for the three months ended June 30, 2026) was only #3 (fixed) and #4 (semantic).

The table shows the best 5 **after** reranking. Compare `rerank_rank` with `retrieve_rank` to see what moved. Change `QUESTION` to try others.

In [6]:
QUESTION = "Which loans are on non-accrual status?"
COLUMNS = ["rerank_rank", "retrieve_rank", "rerank_score", "distance", "chunk_id", "type", "preview"]

results = {name: search(QUESTION, name) for name in STRATEGIES}
for name, r in results.items():
    print(f"\n=== {name}: best {KEEP} of {RETRIEVE_K} after reranking ===")
    display(r.head(KEEP)[COLUMNS])


=== fixed: best 5 of 20 after reranking ===


,rerank_rank,retrieve_rank,rerank_score,distance,chunk_id,type,preview
0,1,1,8.551,0.271,fixed-0031,text,Loans are generally placed on non-accrual status when principal or interest payments are past due 30 days ...
1,2,2,6.112,0.396,fixed-0063,text,"Except as described in the table above listing the Company’s loans held for investment portfolio, as of Ju..."
2,3,14,2.956,0.489,fixed-0060,text,"(14) In June 2026, the Company and the borrower entered into a modification and extension agreement to, am..."
3,4,16,-4.916,0.489,fixed-0075,text,Accrued Interest Receivable The Company elected not to measure a CECL Reserve on accrued interest receiva...
4,5,11,-7.107,0.482,fixed-0044,text,while the unamortized balance on the (i) Secured Funding Agreements (each individually defined in Note 6 i...



=== semantic: best 5 of 20 after reranking ===


,rerank_rank,retrieve_rank,rerank_score,distance,chunk_id,type,preview
0,1,1,7.307,0.353,semantic-0031,text,will be collected in full. Accrued and unpaid interest is generally reversed against interest income in th...
1,2,2,5.374,0.423,semantic-0059,text,"new borrower equity related to additional capital contributions. As of June 30, 2026, the subordinated C-N..."
2,3,7,2.664,0.463,semantic-0061,text,"(1) In March 2026, the Company received a discounted payoff on a senior mortgage loan with outstanding pri..."
3,4,19,-1.927,0.510,semantic-0056,text,"outstanding principal balance of $7.9 million as of June 30, 2026. In January 2026, amortization began on ..."
4,5,12,-4.261,0.487,semantic-0074,text,The Company elected not to measure a CECL Reserve on accrued interest receivable due to the Company’s poli...


Read the full text of any chunk in the table (set the strategy and rerank rank):

In [7]:
show_chunk(results["fixed"], rank=1)

=== rerank #1 · retrieve #1 · fixed-0031 · Part I · Item 1. Consolidated Financial Statements ===
Loans are generally placed on non-accrual status when principal or interest payments are past due 30 days or more or when there is reasonable doubt that principal or interest will be collected in full. Accrued and unpaid interest is generally reversed against interest income in the period the loan is placed on non-accrual status. Interest payments received on non-accrual loans may be recognized as income or applied to reduce loan Carrying Value depending upon management’s judgment regarding the borrower’s ability to make pending principal and interest payments. Non-accrual loans are restored to accrual status when past due principal and interest are paid and, in management’s judgment, are likely to remain current. The Company may make exceptions to placing a loan on non-accrual status if the loan has sufficient collateral value and is in the process of collection.


## A few more analyst questions

The top chunk for each question, **before** (best by vector distance) and **after** reranking. Where they differ, the reranker changed its mind. Read the previews and judge which one actually answers the question.

These are only a first look, not a score; the Evaluate stage will measure it properly with known answers.

In [8]:
QUESTIONS = [
    "What was the net interest margin this quarter?",
    "What was the total carrying value of loans held for investment at June 30, 2026?",
    "How large is the current expected credit loss (CECL) reserve?",
    "What dividend per share was declared for the quarter?",
    "Which loans are on non-accrual status?",
]

rows = []
for q in QUESTIONS:
    for name in STRATEGIES:
        r = search(q, name)
        before = r.loc[r["retrieve_rank"] == 1].iloc[0]
        after = r.iloc[0]
        rows.append({
            "question": q,
            "strategy": name,
            "changed": before.chunk_id != after.chunk_id,
            "before": f"{before.chunk_id}: {before.preview}",
            "after": f"{after.chunk_id} (was #{after.retrieve_rank}): {after.preview}",
        })

pd.DataFrame(rows)

,question,strategy,changed,before,after
0,What was the net interest margin this quarter?,fixed,True,fixed-0280: Table: Net Interest Margin and Interest Expense / Net interest margin in the Company’s consoli...,"fixed-0182 (was #5): For the six months ended June 30, 2026 and 2025, net interest margin was approximatel..."
1,What was the net interest margin this quarter?,semantic,True,semantic-0286: Table: Net Interest Margin and Interest Expense / Net interest margin in the Company’s cons...,"semantic-0186 (was #6): 2025. For the six months ended June 30, 2026 and 2025, net interest margin was app..."
2,"What was the total carrying value of loans held for investment at June 30, 2026?",fixed,True,fixed-0049: The Company’s investments in loans held for investment are accounted for at amortized cost. Th...,fixed-0063 (was #6): Except as described in the table above listing the Company’s loans held for investmen...
3,"What was the total carrying value of loans held for investment at June 30, 2026?",semantic,True,"semantic-0047: The following tables summarize the Company’s loans held for investment as of June 30, 2026 ...",semantic-0314 (was #14): Table: Other than as set forth in Note 3 to our consolidated financial statements...
4,How large is the current expected credit loss (CECL) reserve?,fixed,False,fixed-0289: Table: Current Expected Credit Loss Reserve for Funded Loan Commitments / Activity related to ...,fixed-0289 (was #1): Table: Current Expected Credit Loss Reserve for Funded Loan Commitments / Activity re...
5,How large is the current expected credit loss (CECL) reserve?,semantic,False,semantic-0295: Table: Current Expected Credit Loss Reserve for Funded Loan Commitments / Activity related ...,semantic-0295 (was #1): Table: Current Expected Credit Loss Reserve for Funded Loan Commitments / Activity...
6,What dividend per share was declared for the quarter?,fixed,False,fixed-0161: The Company’s board of directors declared a regular cash dividend of $0.15 per common share fo...,fixed-0161 (was #1): The Company’s board of directors declared a regular cash dividend of $0.15 per common...
7,What dividend per share was declared for the quarter?,semantic,True,semantic-0160: SUBSEQUENT EVENTS The Company’s management has evaluated subsequent events through the dat...,semantic-0184 (was #3): Management’s Discussion and Analysis of Financial Condition and Results of Operati...
8,Which loans are on non-accrual status?,fixed,False,fixed-0031: Loans are generally placed on non-accrual status when principal or interest payments are past ...,fixed-0031 (was #1): Loans are generally placed on non-accrual status when principal or interest payments ...
9,Which loans are on non-accrual status?,semantic,False,semantic-0031: will be collected in full. Accrued and unpaid interest is generally reversed against intere...,semantic-0031 (was #1): will be collected in full. Accrued and unpaid interest is generally reversed again...
